## Concept 2 — LCEL RAG + Retriever Abstraction + MMR

### What is it?
Two improvements over Simple RAG:
1. **Retriever abstraction** — `vectorstore.as_retriever()` replaces manual similarity search. The LCEL `|` pipe chain handles retrieval automatically.
2. **MMR (Maximal Marginal Relevance)** — instead of returning the top K most similar docs (which may all say the same thing), MMR picks docs that are both *relevant* and *diverse* from each other.

### Pipeline
```
Query → Retriever (MMR) → Diverse Top K Docs → LCEL Chain → LLM → Answer
```

### Why MMR?
Without MMR: top 3 docs may all be about the same sentence in different chunks.
With MMR: top 3 docs cover different aspects of the topic.

### Limitation (what Concept 3 fixes)
All retrieved docs are treated equally — doc ranked 1st and doc ranked 5th both go to LLM with same weight.
→ Concept 3 adds reranking to score and filter docs by actual relevance.

In [1]:
print("Hi")

Hi


In [2]:
import sys
sys.path.insert(0, '.')

from RAGutils import setup, format_docs, TEST_QUERIES, run_queries
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser

chunks, vectorstore, embeddings, llm, prompt = setup()

USER_AGENT environment variable not set, consider setting it to identify your requests.


Loading documents...
Splitting into chunks...
Creating embeddings and vector store...

Ready: 1 page(s) -> 11 chunks -> vector store created
LLM: gpt-4o-mini | Embeddings: text-embedding-3-small


### Step 1 — Create Retriever with MMR
`search_type='mmr'` enables Maximal Marginal Relevance.
`fetch_k=10` fetches 10 candidates, then MMR picks the best 3 diverse ones.

In [3]:
retriever = vectorstore.as_retriever(
    search_type="mmr",
    search_kwargs={"k": 3, "fetch_k": 10}
)

# Test retrieval directly
query = TEST_QUERIES[0]
docs = retriever.invoke(query)
print(f"Query: {query}")
print(f"Retrieved {len(docs)} diverse docs via MMR")

Query: What is LangSmith persistence?
Retrieved 3 diverse docs via MMR


### Step 2 — Build LCEL Chain
LCEL uses `|` pipe operator to chain operations. The retriever feeds directly into the prompt.

In [4]:
rag_chain = (
    {
        "context":  retriever | format_docs,
        "question": RunnablePassthrough()
    }
    | prompt
    | llm
    | StrOutputParser()
)

### Step 3 — Wrap into Function
The chain replaces all the manual wiring from Concept 1.

In [6]:
def lcel_rag(query):
    return rag_chain.invoke(query)

### Test — Standard Queries

In [7]:
run_queries(lcel_rag, label="Concept 2 — LCEL RAG + MMR")


 Concept 2 — LCEL RAG + MMR — Test Results

Q: What is LangSmith persistence?
A: LangSmith persistence refers to the storage mechanisms used to retain data across various components of the agent server. This includes core resource data, checkpoints that ensure durability of runs, and long-term memory (store) that allows agents to retain information between conversations. By default, these are stored in PostgreSQL, but there is an option to switch to MongoDB or a custom implementation.
----------------------------------------

Q: What are checkpoints in LangSmith?
A: Checkpoints in LangSmith are snapshots of graph execution state written at each step, making runs durable. If a worker is interrupted, the run can resume from the last checkpoint instead of starting over. The durability mode controls checkpoint frequency, with async (default) writing after each step and exit storing only the final state.
----------------------------------------

Q: Explain LangSmith deployment
A: LangSmith

### Bonus — Multi-Retriever
Query multiple data sources in parallel and combine results.
Toggle `USE_R2 = True` to enable the second source.

In [9]:
from RAGutils import load_docs, split_docs, create_vectorstore, get_embeddings
from langchain_core.runnables import RunnableParallel

embeddings2 = get_embeddings()

# Load second data source
USE_R2 = True  # set True to enable

retriever1 = vectorstore.as_retriever(search_kwargs={"k": 2})

if USE_R2:
    docs2  = load_docs("https://docs.langchain.com/langsmith/platform-setup")
    chunks2 = split_docs(docs2)
    vs2    = create_vectorstore(chunks2, embeddings2)
    retriever2 = vs2.as_retriever(search_kwargs={"k": 2})
    active_retrievers = {"r0": retriever1, "r1": retriever2}
else:
    active_retrievers = {"r0": retriever1}

multi_retriever = RunnableParallel(active_retrievers)

def format_multi(inputs):
    all_docs = []
    for docs in inputs.values():
        all_docs.extend(docs)
    return format_docs(all_docs)

multi_chain = (
    {"context": multi_retriever | format_multi, "question": RunnablePassthrough()}
    | prompt | llm | StrOutputParser()
)

print(multi_chain.invoke(TEST_QUERIES[2]))

LangSmith can be deployed in two modes: Cloud and Self-hosted. 

1. **Cloud**: This mode is fully managed by LangChain, offering quick setup, managed infrastructure, and ease of use.

2. **Self-hosted**: Available on the Enterprise plan, this mode allows you to manage the full stack within your own infrastructure, providing full control and data isolation.

In both deployment options, you can deploy agents and have access to observability, evaluation, and prompt engineering features.
